# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"

# Day we're updating data
update_date = "05-14-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Get rid of missing dates; they won't be counted anyway
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]

# # Find only >= 2024 to start
# metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2021, 11, 1).strftime("%Y-%m-%d")] # Note that those with only years will default to today

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2025, 4, 14).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 5, 14).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025

571


In [3]:
# Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["D1.1"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"] == genotypes[0]]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

307


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,Genotype
8437,SRR33125061,WGS,273.94,163537255,PRJNA1207547,SAMN47941507,Viral,59516737,USDA-NVSL,2025,...,2025-04-14 15:05:12,1,25-005864-001,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS24712489,False,NaN,D1.1
8446,SRR33125070,WGS,262.98,203399754,PRJNA1207547,SAMN47941499,Viral,73253192,USDA-NVSL,2025,...,2025-04-14 15:05:43,1,25-006102-001,SRP557452,NaN,cloacal swab,SRS24712480,False,NaN,D1.1
8449,SRR33125073,WGS,253.20,121517440,PRJNA1207547,SAMN47941443,Viral,42696975,USDA-NVSL,2025,...,2025-04-14 15:05:49,1,25-006015-001,SRP557452,NaN,swab,SRS24712477,False,NaN,D1.1
8450,SRR33125074,WGS,148.23,102424344,PRJNA1207547,SAMN47941496,Viral,37541921,USDA-NVSL,2025,...,2025-04-14 15:03:39,1,25-006328-001,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712476,False,NaN,D1.1
8451,SRR33125075,WGS,240.24,81731522,PRJNA1207547,SAMN47941495,Viral,28473828,USDA-NVSL,2025,...,2025-04-14 15:05:28,1,25-005882-002,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS24712475,False,NaN,D1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8952,SRR33369829,WGS,147.72,146740267,PRJNA1102327,SAMN48199900,Viral,59125311,USDA-NVSL,2025,...,2025-04-29 13:17:57,1,25-011907-002,SRP503016,NaN,"MILK, BULK TANK",SRS24885416,False,NaN,D1.1
8953,SRR33369830,WGS,148.50,182024260,PRJNA1102327,SAMN48199899,Viral,74408243,USDA-NVSL,2024,...,2025-04-29 13:17:55,1,24-036208-002,SRP503016,NaN,milk,SRS24885415,False,NaN,D1.1
8954,SRR33369831,WGS,148.39,131993112,PRJNA1102327,SAMN48199890,Viral,54162419,USDA-NVSL,2024,...,2025-04-29 13:17:56,1,24-036138-002,SRP503016,NaN,"MILK, BULK TANK",SRS24885414,False,NaN,D1.1
8955,SRR33369832,WGS,146.48,232073985,PRJNA1102327,SAMN48199889,Viral,92689954,USDA-NVSL,2024,...,2025-04-29 13:17:57,1,24-036138-001,SRP503016,NaN,"MILK, BULK TANK",SRS24885413,False,NaN,D1.1


In [5]:
# # Get specific geolocation from genbank_mapping.tsv

# genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
# genbank_mapping["Run"] = genbank_mapping["sra_run"]
# genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
# genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# print(genbank_mapping["name_state"])
# print(len(metadata_genbank))
# display(metadata_genbank) # Maybe there is no state information since 3/18/2025?

In [6]:
# If no states

metadata_genbank = metadata

metadata_genbank["name_state"] = "USA"

## Get and save collection date

In [7]:

# # Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [8]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv")
# os.chdir(temp_files)

# # Get only updated dates

# unknown_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] == "2024") | (metadata_genbank["Collection_Date_Specific"] == "2025")] # Dates we don't have
# known_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] != "2024") & (metadata_genbank["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# # Get new dates also 
# # new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata_genbank[["Collection_Date_Specific"]])

# display(metadata_genbank)

In [9]:
# If no collection dates

metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [10]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['cat', 'rock goose', 'american crow', 'american black duck', 'snowy owl', 'cattle', 'guineafowl', 'mute swan', 'skunk', 'red-shouldered hawk', 'mallard', 'herring gull', 'green-winged teal', 'goose', 'western sandpiper', "ross's goose", 'barn owl', 'peregrine', 'duck', 'great horned owl', 'turkey vulture', 'mallard x american black duck hybrid', 'sharp-shinned hawk', 'bald eagle', 'red-tailed hawk', 'snow goose', 'vulture', 'western gull', 'hooded merganser', 'turkey', 'gadwall', 'great black-backed gull', 'snowy egret', 'swan', 'chicken', 'black vulture', 'sandhill crane']
['rock goose', "ross's goose", 'sharp-shinned hawk', 'snowy egret']
                 avian               cattle        feline   other_mammal  \
0     great_horned_owl            dairy_cow           cat     deer mouse   
1         common_raven               cattle  domestic_cat    house_mouse   
2        cooper's_hawk  cattle milk product     feral_cat          skunk   
3         coopers_hawk          bovine_milk   

In [11]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [12]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

In [13]:
print(metadata_genbank)

              Run Assay Type  AvgSpotLen        Bases    BioProject  \
8437  SRR33125061        WGS      273.94  163537255.0  PRJNA1207547   
8446  SRR33125070        WGS      262.98  203399754.0  PRJNA1207547   
8449  SRR33125073        WGS      253.20  121517440.0  PRJNA1207547   
8450  SRR33125074        WGS      148.23  102424344.0  PRJNA1207547   
8451  SRR33125075        WGS      240.24   81731522.0  PRJNA1207547   
...           ...        ...         ...          ...           ...   
302           NaN        NaN         NaN          NaN           NaN   
303           NaN        NaN         NaN          NaN           NaN   
304           NaN        NaN         NaN          NaN           NaN   
305           NaN        NaN         NaN          NaN           NaN   
306           NaN        NaN         NaN          NaN           NaN   

         BioSample BioSampleModel       Bytes Center Name Collection_Date  \
8437  SAMN47941507          Viral  59516737.0   USDA-NVSL            2

## Make FASTA files

In [14]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

TypeError: 'in <string>' requires string as left operand, not float

In [15]:
# print(fasta_files.keys())

In [16]:
# Create fasta files 

os.chdir(originals + "complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/")
names = []
for pair in fasta_files.keys():
    output_path = originals + "complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/" + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names))

>A/HERRING GULL/USA/25-005864-001/2025|H5N1|2025|avian|D1.1
>A/GREAT HORNED OWL/USA/25-006102-001/2025|H5N1|2025|avian|D1.1
>A/BALD EAGLE/USA/25-006015-001/2025|H5N1|2025|avian|D1.1
>A/GREAT BLACK-BACKED GULL/USA/25-006328-001/2025|H5N1|2025|avian|D1.1
>A/GADWALL/USA/25-005882-002/2025|H5N1|2025|avian|D1.1
>A/BARN OWL/USA/25-004498-003/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-006120-004/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-006120-003/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-005756-005/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-005756-004/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-005754-005/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-005754-004/2025|H5N1|2025|avian|D1.1
>A/BLACK VULTURE/USA/25-004339-009/2025|H5N1|2025|avian|D1.1
>A/AMERICAN CROW/USA/25-004493-001/2025|H5N1|2025|avian|D1.1
>A/WESTERN SANDPIPER/USA/25-004497-006/2025|H5N1|2025|avian|D1.1
>A/WESTERN GULL/USA/25-004499-005/2025|H5N1|2025|avian|D1.1
>A/BALD EAGLE/USA/25-00

## De-Duplication

In [17]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID/complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1_northa/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [18]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(originals + "complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/")

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [19]:
os.chdir(downloads)
dfs_gisaid["B3.13_HA"].to_csv

AttributeError: 'list' object has no attribute 'to_csv'

In [20]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
defaultdict(<class 'list'>, {'D1.1_HA': [    isolate_partial                                        full_header  \
0        005864-001  >A/HERRING GULL/USA/25-005864-001/2025|H5N1|20...   
1        006102-001  >A/GREAT HORNED OWL/USA/25-006102-001/2025|H5N...   
2        006015-001  >A/BALD EAGLE/USA/25-006015-001/2025|H5N1|2025...   
3        006328-001  >A/GREAT BLACK-BACKED GULL/USA/25-006328-001/2...   
4        005882-002  >A/GADWALL/USA/25-005882-002/2025|H5N1|2025|av...   
..              ...                                                ...   
302      011907-002  >A/CATTLE/USA/25-011907-002/2025|H5N1|2025|cat...   
303      036208-002  >A/CATTLE/USA/24-036208-002/2024|H5N1|2024|cat...   
304      036138-002  >A/CATTLE/USA/24-036138-002/2024|H5N1|2024|cat...   
305      036138-001  >A/CATTLE/USA/24-036138-001/2024|H5N1|2024|cat...   
306      009630-002  >A/TURKEY/USA/25-009630-002/2025|H5N1|2025|avi...   

    

In [21]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

9
8


In [22]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                gisaid_df = dfs_gisaid[gisaid_key][0]
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df)*8)
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                print(len(full_df.loc[full_df.duplicated(subset="isolate_partial")])*8)
                full_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")
                print("len deduplicated:", len(full_df)*8)
                full_dfs[andersen_key].append(full_df)
            # else:
            #     gisaid.add(gisaid_key)
            #     andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664
len full df: 5504
840
len deduplicated: 4664


In [23]:
# If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             # full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)

## Create FASTA files combining Andersen and GISAID

In [24]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
